[data dictionary](https://www.nyc.gov/assets/tlc/downloads/pdf/data_dictionary_trip_records_yellow.pdf)

In [1]:
import pandas as pd
import numpy as np
from sklearn.feature_extraction import DictVectorizer
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error
import pickle
import xgboost as xgb
import optuna

# MLflow
import mlflow

mlflow.set_experiment('nyt-taxi-duration')

2026/06/03 15:33:59 INFO mlflow.store.db.utils: Creating initial MLflow database tables...
2026/06/03 15:33:59 INFO mlflow.store.db.utils: Updating database tables
2026/06/03 15:34:00 INFO mlflow.tracking.fluent: Experiment with name 'nyt-taxi-duration' does not exist. Creating a new experiment.


<Experiment: artifact_location='/Users/mattfelici/personal/learning/mlops-zoomcamp/03-orchestration/mlruns/1', creation_time=1780493640815, experiment_id='1', last_update_time=1780493640815, lifecycle_stage='active', name='nyt-taxi-duration', tags={}, trace_location=None, workspace='default'>

In [2]:
# df = pd.read_parquet('https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet')
df = pd.read_parquet('../data/yellow_tripdata_2026-01.parquet', engine='fastparquet')

print(df.shape)
print(df.dtypes)
df.head()

(3724889, 20)
VendorID                          int32
tpep_pickup_datetime     datetime64[us]
tpep_dropoff_datetime    datetime64[us]
passenger_count                 float64
trip_distance                   float64
RatecodeID                      float64
store_and_fwd_flag               object
PULocationID                      int32
DOLocationID                      int32
payment_type                      int64
fare_amount                     float64
extra                           float64
mta_tax                         float64
tip_amount                      float64
tolls_amount                    float64
improvement_surcharge           float64
total_amount                    float64
congestion_surcharge            float64
Airport_fee                     float64
cbd_congestion_fee              float64
dtype: object


,VendorID,tpep_pickup_datetime,tpep_dropoff_datetime,passenger_count,trip_distance,RatecodeID,store_and_fwd_flag,PULocationID,DOLocationID,payment_type,fare_amount,extra,mta_tax,tip_amount,tolls_amount,improvement_surcharge,total_amount,congestion_surcharge,Airport_fee,cbd_congestion_fee
0,2,2026-01-01 00:54:04,2026-01-01 00:59:37,1.0,0.97,1.0,N,239,238,1,7.2,1.00,0.5,3.66,0.0,1.0,15.86,2.5,0.0,0.00
1,1,2026-01-01 00:34:04,2026-01-01 00:39:47,0.0,0.90,1.0,N,163,162,2,7.9,4.25,0.5,0.00,0.0,1.0,13.65,2.5,0.0,0.75
2,1,2026-01-01 00:57:06,2026-01-01 01:05:59,0.0,1.40,1.0,N,43,237,1,10.7,4.25,0.5,2.50,0.0,1.0,18.95,2.5,0.0,0.75
3,2,2026-01-01 00:15:22,2026-01-01 00:58:10,4.0,5.58,1.0,N,142,209,1,38.7,1.00,0.5,11.11,0.0,1.0,55.56,2.5,0.0,0.75
4,2,2026-01-01 00:27:13,2026-01-01 00:40:43,0.0,2.16,1.0,N,88,144,1,13.5,1.00,0.5,3.85,0.0,1.0,23.10,2.5,0.0,0.75


In [3]:
def feature_engineering(df):
    # Duration
    df['duration'] = (df['tpep_dropoff_datetime'] - df['tpep_pickup_datetime']).dt.total_seconds() / 60
    df = df.loc[df['duration'].between(1, 60)]
    
    # Date info
    df['dow'] = df['tpep_pickup_datetime'].dt.day_name()
    df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour

    # Pairing locations
    df['locationID'] = df['PULocationID'].astype(str) + '_' + df['DOLocationID'].astype(str)

    return df

def preprocessing(df, categorical, numerical, tgt='duration'):

    dv = DictVectorizer(separator='_')
    tmp = df[categorical + numerical]
    tmp[categorical] = tmp[categorical].astype(str)
    y = df[tgt]
    
    X_train, X_val, y_train, y_val = train_test_split(tmp, y, train_size=0.8)
    X_train = dv.fit_transform(X_train.to_dict(orient='records'))

    X_val = dv.transform(X_val.to_dict(orient='records'))
    
    return X_train, X_val, y_train, y_val, dv


def complete_preparation(df, categorical, numerical, tgt='duration'):

    df = feature_engineering(df)
    return preprocessing(df, categorical, numerical, tgt)

In [4]:
# Feature list
# categorical = ['PULocationID', 'DOLocationID', 'dow', 'pickup_hour']
categorical = ['locationID', 'dow', 'pickup_hour']
numerical = ['trip_distance']
tgt = 'duration'

X_train, X_val, y_train, y_val, dv = complete_preparation(df, categorical, numerical, tgt=tgt)

/var/folders/ty/vy_xnc9503lgj7fc9h6qd5m80000gn/T/ipykernel_83784/2610881621.py:7: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['dow'] = df['tpep_pickup_datetime'].dt.day_name()
/var/folders/ty/vy_xnc9503lgj7fc9h6qd5m80000gn/T/ipykernel_83784/2610881621.py:8: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['pickup_hour'] = df['tpep_pickup_datetime'].dt.hour
/var/folders/ty/vy_xnc9503lgj7fc9h6qd5m80000gn/T/ipykernel_83784/2610881621.py:11: SettingWithCopyWarning: 
A value is trying to be set on a copy of

In [5]:
train = xgb.DMatrix(X_train, label=y_train.reset_index(drop=True))
val = xgb.DMatrix(X_val, label=y_val.reset_index(drop=True))

In [7]:
type(y_val)

pandas.core.series.Series

In [8]:
params = {
    'max_depth': 12,
    'learning_rate': 1e-2,
    'objective': 'reg:squarederror',
    'seed': 1123
}

booster = xgb.train(
    params=params,
    dtrain=train,
    num_boost_round=100,
    evals=[(val, 'validation')],
    early_stopping_rounds=50
)

[0]	validation-rmse:10.58508
[1]	validation-rmse:10.51461
[2]	validation-rmse:10.44510
[3]	validation-rmse:10.37651
[4]	validation-rmse:10.30882
[5]	validation-rmse:10.24205
[6]	validation-rmse:10.17619
[7]	validation-rmse:10.11123
[8]	validation-rmse:10.04709
[9]	validation-rmse:9.98382
[10]	validation-rmse:9.92139
[11]	validation-rmse:9.85987
[12]	validation-rmse:9.79915
[13]	validation-rmse:9.73922
[14]	validation-rmse:9.68015
[15]	validation-rmse:9.62185
[16]	validation-rmse:9.56433
[17]	validation-rmse:9.50760
[18]	validation-rmse:9.45163
[19]	validation-rmse:9.39644
[20]	validation-rmse:9.34201
[21]	validation-rmse:9.28836
[22]	validation-rmse:9.23552
[23]	validation-rmse:9.18339
[24]	validation-rmse:9.13189
[25]	validation-rmse:9.08116
[26]	validation-rmse:9.03102
[27]	validation-rmse:8.98159
[28]	validation-rmse:8.93285
[29]	validation-rmse:8.88487
[30]	validation-rmse:8.83747
[31]	validation-rmse:8.79082
[32]	validation-rmse:8.74475
[33]	validation-rmse:8.69928
[34]	validation

In [9]:
type(booster)

xgboost.core.Booster

In [ ]:
# single run

with mlflow.start_run(run_name='XGB-training') as child_run:
    mlflow.set_tag('dev', 'mattfelici')
    mlflow.log_param('model_type', 'XGBoost')

    params = {
        'max_depth': 12,
        'learning_rate': 1e-2,
        'objective': 'reg:squarederror',
        'seed': 1123
    }
    
    mlflow.log_params(params)
    booster = xgb.train(
        params=params,
        dtrain=train,
        num_boost_round=100,
        evals=[(val, 'validation')],
        early_stopping_rounds=50
    )
    pred_train = booster.predict(train)
    rmse_train = root_mean_squared_error(y_train, pred_train)
    mlflow.log_metric('RMSE_train', rmse_train)
    
    pred_val = booster.predict(val)
    rmse_val = root_mean_squared_error(y_val, pred_val)
    mlflow.log_metric('RMSE_val', rmse_val) 

In [51]:
def optimize(trial):

    with mlflow.start_run(nested=True, run_name=f'trial_{trial.number}') as child_run:
        mlflow.set_tag('dev', 'mattfelici')
        mlflow.log_param('model_type', 'XGBoost')
        mlflow.set_tag('model-optimization', 'hyperopt')

        params = {
            'max_depth': trial.suggest_int('max_depth', 4, 25),
            'learning_rate': trial.suggest_float('learning_rate', 1e-2, 1e0, log=True),
            'reg_alpha': trial.suggest_float('reg_alpha', 1e-5, 1e-1, log=True),
            'reg_lambda': trial.suggest_float('reg_lambda', 1e-6, 1e-1, log=True),
            'min_child_weight': trial.suggest_float('min_child_weight', 1e-1, 1e3, log=True),
            'objective': 'reg:squarederror',
            'seed': 42
        }
        
        # mlflow.xgboost.autolog()
        mlflow.log_params(params)
        booster = xgb.train(
            params=params,
            dtrain=train,
            num_boost_round=100,
            evals=[(val, 'validation')],
            early_stopping_rounds=50
        )
        pred_train = booster.predict(train)
        rmse_train = root_mean_squared_error(y_train, pred_train)
        mlflow.log_metric('RMSE_train', rmse_train)
        
        pred_val = booster.predict(val)
        rmse_val = root_mean_squared_error(y_val, pred_val)
        mlflow.log_metric('RMSE_val', rmse_val) 

        trial.set_user_attr("run_id", child_run.info.run_id)

        return rmse_val

In [52]:
with mlflow.start_run(run_name='XGB hyperparam opt') as run:

    n_trials = 10
    mlflow.log_param('N trials', n_trials)
    
    study = optuna.create_study(direction="minimize")
    study.optimize(optimize, n_trials=n_trials)
    
    # Log the best trial and its run ID
    mlflow.log_params(study.best_trial.params)

    booster = xgb.train(
        params=study.best_trial.params,
        dtrain=train,
        num_boost_round=100,
        evals=[(val, 'validation')],
        early_stopping_rounds=50
    )
    pred_train = booster.predict(train)
    rmse_train = root_mean_squared_error(y_train, pred_train)
    mlflow.log_metric('RMSE_train', rmse_train)
    
    pred_val = booster.predict(val)
    rmse_val = root_mean_squared_error(y_val, pred_val)
    mlflow.log_metric('RMSE_val', rmse_val) 

    with open('dict_vectorizer_fit.b', 'wb') as f_out:
        pickle.dump(dv, f_out)
    mlflow.log_artifact('dict_vectorizer_fit.b', artifact_path='preprocessors')
    
    mlflow.xgboost.log_model(booster, name='xgb_model')
    if best_run_id := study.best_trial.user_attrs.get('run_id'):
        mlflow.log_param('best_child_run_id', best_run_id)

[I 2026-05-19 11:02:06,339] A new study created in memory with name: no-name-1bc7133f-4281-4ceb-aff8-2ba629646365


[0]	validation-rmse:6.50193
[1]	validation-rmse:5.86984
[2]	validation-rmse:5.75987
[3]	validation-rmse:5.72278
[4]	validation-rmse:5.69936
[5]	validation-rmse:5.69049
[6]	validation-rmse:5.68329
[7]	validation-rmse:5.67758
[8]	validation-rmse:5.67181
[9]	validation-rmse:5.66169
[10]	validation-rmse:5.65356
[11]	validation-rmse:5.64747
[12]	validation-rmse:5.64310
[13]	validation-rmse:5.63883
[14]	validation-rmse:5.62882
[15]	validation-rmse:5.62499
[16]	validation-rmse:5.62130
[17]	validation-rmse:5.61813
[18]	validation-rmse:5.61457
[19]	validation-rmse:5.61158
[20]	validation-rmse:5.60826
[21]	validation-rmse:5.60543
[22]	validation-rmse:5.60262
[23]	validation-rmse:5.59941
[24]	validation-rmse:5.59572
[25]	validation-rmse:5.59366
[26]	validation-rmse:5.59156
[27]	validation-rmse:5.58740
[28]	validation-rmse:5.58565
[29]	validation-rmse:5.58334
[30]	validation-rmse:5.58169
[31]	validation-rmse:5.57754
[32]	validation-rmse:5.57595
[33]	validation-rmse:5.57371
[34]	validation-rmse:5.5

[I 2026-05-19 11:02:58,187] Trial 0 finished with value: 5.526896043466545 and parameters: {'max_depth': 23, 'learning_rate': 0.6863627697279705, 'reg_alpha': 0.00018218052816230037, 'reg_lambda': 0.01693166124798123, 'min_child_weight': 534.8075619385314}. Best is trial 0 with value: 5.526896043466545.


[0]	validation-rmse:8.37169
[1]	validation-rmse:7.16995
[2]	validation-rmse:6.56387
[3]	validation-rmse:6.26912
[4]	validation-rmse:6.11128
[5]	validation-rmse:6.02036
[6]	validation-rmse:5.96843
[7]	validation-rmse:5.92818
[8]	validation-rmse:5.89574
[9]	validation-rmse:5.87766
[10]	validation-rmse:5.86252
[11]	validation-rmse:5.83647
[12]	validation-rmse:5.82690
[13]	validation-rmse:5.81109
[14]	validation-rmse:5.80532
[15]	validation-rmse:5.80062
[16]	validation-rmse:5.79519
[17]	validation-rmse:5.79052
[18]	validation-rmse:5.78383
[19]	validation-rmse:5.77781
[20]	validation-rmse:5.77384
[21]	validation-rmse:5.76998
[22]	validation-rmse:5.76623
[23]	validation-rmse:5.75785
[24]	validation-rmse:5.75419
[25]	validation-rmse:5.75085
[26]	validation-rmse:5.74740
[27]	validation-rmse:5.74409
[28]	validation-rmse:5.74068
[29]	validation-rmse:5.73750
[30]	validation-rmse:5.73408
[31]	validation-rmse:5.72845
[32]	validation-rmse:5.72537
[33]	validation-rmse:5.72261
[34]	validation-rmse:5.7

[I 2026-05-19 11:03:38,665] Trial 1 finished with value: 5.575710595714496 and parameters: {'max_depth': 9, 'learning_rate': 0.35484018824959207, 'reg_alpha': 8.2226421613827e-05, 'reg_lambda': 3.9485274997544754e-05, 'min_child_weight': 5.047857175908173}. Best is trial 0 with value: 5.526896043466545.


...

[0]	validation-rmse:10.56847
[1]	validation-rmse:10.48001
[2]	validation-rmse:10.39294
[3]	validation-rmse:10.30721
[4]	validation-rmse:10.22283
[5]	validation-rmse:10.13978
[6]	validation-rmse:10.05804
[7]	validation-rmse:9.97760
[8]	validation-rmse:9.89842
[9]	validation-rmse:9.82050
[10]	validation-rmse:9.74379
[11]	validation-rmse:9.66829
[12]	validation-rmse:9.59406
[13]	validation-rmse:9.52099
[14]	validation-rmse:9.44918
[15]	validation-rmse:9.37849
[16]	validation-rmse:9.30891
[17]	validation-rmse:9.24053
[18]	validation-rmse:9.17323
[19]	validation-rmse:9.10711
[20]	validation-rmse:9.04196
[21]	validation-rmse:8.97802
[22]	validation-rmse:8.91501
[23]	validation-rmse:8.85322
[24]	validation-rmse:8.79235
[25]	validation-rmse:8.73253
[26]	validation-rmse:8.67378
[27]	validation-rmse:8.61598
[28]	validation-rmse:8.55917
[29]	validation-rmse:8.50338
[30]	validation-rmse:8.44850
[31]	validation-rmse:8.39458
[32]	validation-rmse:8.34159
[33]	validation-rmse:8.28952
[34]	validation-r

[I 2026-05-19 11:36:45,978] Trial 8 finished with value: 6.329489454519392 and parameters: {'max_depth': 23, 'learning_rate': 0.01196708599307545, 'reg_alpha': 0.02049608298057903, 'reg_lambda': 1.3180153877433372e-05, 'min_child_weight': 13.342635814079724}. Best is trial 2 with value: 5.447248206515293.


[0]	validation-rmse:8.97514
[1]	validation-rmse:7.82661
[2]	validation-rmse:7.06614
[3]	validation-rmse:6.57569
[4]	validation-rmse:6.26348
[5]	validation-rmse:6.06591
[6]	validation-rmse:5.93975
[7]	validation-rmse:5.86005
[8]	validation-rmse:5.80493
[9]	validation-rmse:5.76823
[10]	validation-rmse:5.74018
[11]	validation-rmse:5.72210
[12]	validation-rmse:5.70597
[13]	validation-rmse:5.69382
[14]	validation-rmse:5.68181
[15]	validation-rmse:5.67271
[16]	validation-rmse:5.66415
[17]	validation-rmse:5.65883
[18]	validation-rmse:5.65411
[19]	validation-rmse:5.64954
[20]	validation-rmse:5.64112
[21]	validation-rmse:5.63892
[22]	validation-rmse:5.63657
[23]	validation-rmse:5.63437
[24]	validation-rmse:5.63213
[25]	validation-rmse:5.62985
[26]	validation-rmse:5.62775
[27]	validation-rmse:5.62561
[28]	validation-rmse:5.62329
[29]	validation-rmse:5.62118
[30]	validation-rmse:5.61925
[31]	validation-rmse:5.61744
[32]	validation-rmse:5.61573
[33]	validation-rmse:5.61383
[34]	validation-rmse:5.6

[I 2026-05-19 11:37:48,143] Trial 9 finished with value: 5.5313743713527135 and parameters: {'max_depth': 19, 'learning_rate': 0.2379911694427928, 'reg_alpha': 0.0027519638862466204, 'reg_lambda': 3.5011168357716195e-06, 'min_child_weight': 305.87524935340696}. Best is trial 2 with value: 5.447248206515293.


[0]	validation-rmse:8.20423
[1]	validation-rmse:6.91319
[2]	validation-rmse:6.27986
[3]	validation-rmse:5.97838
[4]	validation-rmse:5.82789
[5]	validation-rmse:5.74981
[6]	validation-rmse:5.70451
[7]	validation-rmse:5.67860
[8]	validation-rmse:5.65905
[9]	validation-rmse:5.64126
[10]	validation-rmse:5.63207
[11]	validation-rmse:5.62381
[12]	validation-rmse:5.61319
[13]	validation-rmse:5.60908
[14]	validation-rmse:5.60553
[15]	validation-rmse:5.60199
[16]	validation-rmse:5.59824
[17]	validation-rmse:5.59463
[18]	validation-rmse:5.59137
[19]	validation-rmse:5.58761
[20]	validation-rmse:5.58415
[21]	validation-rmse:5.58095
[22]	validation-rmse:5.57778
[23]	validation-rmse:5.57275
[24]	validation-rmse:5.56992
[25]	validation-rmse:5.56723
[26]	validation-rmse:5.56452
[27]	validation-rmse:5.56145
[28]	validation-rmse:5.55899
[29]	validation-rmse:5.55625
[30]	validation-rmse:5.55394
[31]	validation-rmse:5.55171
[32]	validation-rmse:5.54879
[33]	validation-rmse:5.54656
[34]	validation-rmse:5.5

### Let's recall the best model

In [18]:
from mlflow.tracking import MlflowClient

client = MlflowClient()

In [57]:
model_name = 'nyc-taxi-regressor'
model_version_alias = 'best_model'

model_info = client.get_model_version_by_alias(model_name, model_version_alias)
run = client.get_run(model_info.run_id)
print(run.data.params)

model_uri = f'models:/{model_name}@{model_version_alias}'
best_model = mlflow.xgboost.load_model(model_uri)

print(best_model)

{'N trials': '10', 'max_depth': '22', 'learning_rate': '0.3539635775310726', 'reg_alpha': '0.00024114931359639884', 'reg_lambda': '0.0015916602556397868', 'min_child_weight': '220.16781134769855', 'best_child_run_id': '51e2f94e205d48a89d207f4628845c9c'}


In [56]:
# Query runs
runs = client.search_runs(
    experiment_ids='1',
    filter_string='metrics.RMSE_val < 6',
    run_view_type=mlflow.entities.ViewType.ACTIVE_ONLY,
    max_results=10,
    order_by=['metrics.RMSE_val ASC']
)

for run in runs:
    print(f'run_id: {run.info.run_id}, run_name: {run.info.run_name}, metric: {run.data.metrics['RMSE_val']}')

run_id: 51e2f94e205d48a89d207f4628845c9c, run_name: trial_2, metric: 5.447248206515293
run_id: 6638d1969c8341abbded2bf39360f63d, run_name: XGB hyperparam opt, metric: 5.447248206515293
run_id: 2234f0ac471347568d78709b47e9dfcf, run_name: trial_7, metric: 5.526833629004594
run_id: fea994f8f67f460f96d145dc76f62448, run_name: trial_0, metric: 5.526896043466545
run_id: 018fbc9efc4f42b3ac3522a0b38fc171, run_name: trial_9, metric: 5.5313743713527135
run_id: 3888975698914d308ec3856124c8b0c6, run_name: trial_6, metric: 5.536921763474685
run_id: 1e368d96efe74126b87c696a89dadef7, run_name: trial_3, metric: 5.575095452400171
run_id: a8ac07c9d3a1488dac3546b9b8078184, run_name: trial_1, metric: 5.575710595714496
run_id: df7b8950c80e4690b1f75f3a96ca3d23, run_name: trial_4, metric: 5.648243205885593
run_id: 43974591fe5a41029a36057c94449e6a, run_name: intelligent-fly-533, metric: 5.670632941857917


In [59]:
# Register model
model_uri = f'run:/{runs[1].info.run_id}/xgb_model'
mlflow.register_model(model_uri, name=model_name)

Registered model 'nyc-taxi-regressor' already exists. Creating a new version of this model...
Created version '4' of model 'nyc-taxi-regressor'.


<ModelVersion: aliases=[], creation_timestamp=1779187808148, current_stage='None', deployment_job_state=None, description=None, last_updated_timestamp=1779187808148, metrics=None, model_id=None, name='nyc-taxi-regressor', params=None, run_id=None, run_link=None, source='run:/6638d1969c8341abbded2bf39360f63d/xgb_model', status='READY', status_message=None, tags={}, user_id=None, version=4, workspace='default'>